In [1]:
%cd ../../../

/Users/hoangle/Food-Waste-Optimization


In [2]:
import re

import numpy as np
import pandas as pd


# Process POS data

In [3]:
paths = [
    "data/raw/pos/Sold lunches.csv",
    "data/raw/pos/Sold lunches Kumpula 6-8 2024.csv",
    "data/raw/pos/Sold lunches Kumpula 9-10 2024.csv",
    "data/raw/pos/Sold lunches Viikuna 2023.csv",
    "data/raw/pos/Sold lunches Viikuna 2024.csv"
]

raw = []
for path in paths:
    df = pd.read_csv(path, delimiter=';')
    df.columns = np.arange(df.shape[1])
    raw.append(df)


pos_raw = pd.concat(raw, ignore_index=True)
pos_raw.head()

/var/folders/pr/8dv_cj95295bxt_hr8hzrmk40000gn/T/ipykernel_46692/1467303410.py:11: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, delimiter=';')


,0,1,2,3,4,5,6
0,2.1.2023,10:31,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",1,"0,9"
1,2.1.2023,10:32,600 Chemicum,Kala,Kalapuikot tillikermaviilikast,1,"1,04"
2,2.1.2023,10:32,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",1,"0,9"
3,2.1.2023,10:35,600 Chemicum,Kala,Kalapuikot tillikermaviilikast,1,"1,04"
4,2.1.2023,10:36,600 Chemicum,Liha,"Uunimakkaraa,sinappikastiketta",2,"1,8"


In [4]:
pos = pos_raw.copy()


# Rename columns
pos.columns = np.arange(pos.shape[1])
pos.rename(
    columns={
        0: 'date',
        1: 'time',
        2: 'restaurant',
        3: 'meal_type',
        4: 'meal',
        5: 'pcs',
        6: 'co2',
    },
    inplace=True
)


# Convert pcs
def _f_conv(s):
    match s:
        case int() | float():
            return float(s)
        case str():
            if re.search(r'\s', s) is None:
                return float(s)
            return np.nan

pos['pcs'] = pos['pcs'].apply(_f_conv)
pos = pos[~pos['pcs'].isna()]


# Map restaurant name
pos['restaurant'] = pos['restaurant'].map({
    '600 Chemicum': 'che', #'chemicum',
    '610 Physicum': 'phy', #'physicum',
    '620 Exactum': 'exa', #'exactum'
    '570 Viikuna': 'vik',
})


# Process date
pos['date'] = pd.to_datetime(pos['date'], format="%d.%m.%Y")


# Process meal_type
pos['meal_type'] = pos['meal_type'].map({
    'Liha': 'meat',
    'Kala': 'fish',
    'Vegaani': 'vegan',
    'Kasvis': 'vegetarian',
    'Kana': 'chicken'
})


# Process meal
pos['meal'] = pos['meal'].str.strip()


# Remove 'take away' meals
pos = pos[~pos['meal'].str.lower().str.contains('take away')]



# Accumulate pcs per day
pos = (
    pos
    .groupby(['date', 'restaurant', 'meal', 'meal_type'])['pcs']
    .sum()
    .reset_index()
)


# Remove POS of leftover
DELTA = 3
pos['date_prev'] = (
    pos
    .sort_values('date')
    .groupby(['meal', 'meal_type', 'restaurant'])['date']
    .shift(1)    
)
pos = pos[
    (pos['date_prev'].isna())
    | ((pos['date'] - pos['date_prev']).dt.days > DELTA)
]
pos.drop(columns=['date_prev'], inplace=True)


pos.head()

,date,restaurant,meal,meal_type,pcs
0,2023-01-02,che,Kalapuikot tillikermaviilikast,fish,78.0
1,2023-01-02,che,Marokkolainen linssipata,vegan,84.0
2,2023-01-02,che,"Uunimakkaraa,sinappikastiketta",meat,165.0
3,2023-01-03,che,Feta-pinaattilasagnette,vegetarian,29.0
4,2023-01-03,che,"Herkkulohipihvit, punajuurimaj",fish,105.0


# Get metadata from `dim_meals` and `dim_meal_names`

In [5]:
path = 'data/processed/phase_4/dim_meal_names.xlsx'

dim_meal_names = pd.read_excel(path)
dim_meal_names.head()

,meal_id,meal
0,9017,"""Butter"" härkäpapua & pähkinää"
1,7201,2023 Härkäpu-sienilasagnette
2,9032,Appelisiini-luomukikhernecurrya
3,9102,Artisokkavugetteja & tuoretomaattisalsaa
4,7010,Aurajuusto-pinaattilasagnette


In [6]:
path = 'data/processed/phase_4/dim_meals.xlsx'

dim_meals = pd.read_excel(path)
dim_meals.head()

,meal_id,meal_type_1,schoolyear,is_kela,is_new,restaurant,meal_type_2,pcs_mean
0,9017,vegan,24-25,True,True,che-exa-vik,vegan-miscellaneous,116.342500
1,7201,vegan,23-24,False,False,NaN,NaN,106.231119
2,9032,vegan,23-24,False,False,NaN,NaN,106.231119
3,9102,vegan,23-24,False,False,NaN,NaN,106.231119
4,7010,vegetarian,24-25,False,False,che-exa-vik,NaN,71.000000


In [7]:
pos = (
    pos
    .merge(dim_meal_names, on='meal', how='left')
    .merge(dim_meals, on='meal_id', how='left')
    .drop(columns=['meal', 'schoolyear', 'is_kela', 'is_new', 'restaurant_y', 'meal_type_2', 'meal_type_1', 'pcs_mean'])
    .rename(columns={'restaurant_x': 'restaurant'})
)

pos.head()

,date,restaurant,meal_type,pcs,meal_id
0,2023-01-02,che,fish,78.0,9500055
1,2023-01-02,che,vegan,84.0,6128
2,2023-01-02,che,meat,165.0,9500160
3,2023-01-03,che,vegetarian,29.0,1270
4,2023-01-03,che,fish,105.0,6156


# Save

In [8]:
pos.to_excel("data/processed/phase_4/dim_pieces_per_dish.xlsx", index=False)